#Step 7: Mapping on the two groups (linguistic_cluster and social_community)

Note that this process is generalizable and can be applied to multiple social communities and not necessairly one like in this example.

In [ ]:
from google.colab import drive
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Fragment/fragmentTry/df_step6_complete.csv')

df

In [ ]:
#explorative analysis
print(f" number of linguistic clusters (excluding noise): {df[df['linguistic_cluster'] != -1]['linguistic_cluster'].nunique()}")
print(f" number of social communities: {df['social_community'].nunique()}")

# Exploring linguistic clusters

In [ ]:
import ast
import collections
import re

#to parse
def safe_parse_history(x):
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return x if isinstance(x, list) else []

#to obtain financial insights
def get_financial_metrics(row):
    history = row['parsed_history']
    transfers = len(history) if isinstance(history, list) else 0

    last_price = 0
    if isinstance(history, list):
        for h in history:
            if isinstance(h, dict) and 'Price' in h:
                try:
                    val = float(h['Price'])
                    if val > 0:
                        last_price = val
                        break
                except:
                    continue

    return pd.Series([transfers, last_price])

#to obtain the top n-grams
def get_top_ngrams(series, n=4, top_k=10):
    text = " ".join(series.astype(str)).lower().replace('@', '')
    text = re.sub(r'[^a-z0-9]', ' ', text)
    tokens = text.split()

    ngram_counts = collections.Counter()

    for token in tokens:
        if len(token) >= n:
            for i in range(len(token) - n + 1):
                ngram = token[i:i+n]
                ngram_counts[ngram] += 1

    if not ngram_counts: return []
    return [f"{k}" for k, v in ngram_counts.most_common(top_k)]

#parse financial data
df['parsed_history'] = df['Ownership history'].apply(safe_parse_history)
df[['transfer_count', 'last_sale_price']] = df.apply(get_financial_metrics, axis=1)
#obtain total volume and save in dataset
total_realized_volume = df['last_sale_price'].sum()

#analyze each cluster singularly, to prepare for mapping
unique_clusters = sorted(df[df['linguistic_cluster'] != -1]['linguistic_cluster'].unique())

for cid in unique_clusters:
    subset = df[df['linguistic_cluster'] == cid]
    size = len(subset)

    #order by value
    subset_sorted = subset.sort_values(by='last_sale_price', ascending=False)

    #economic metrics
    cluster_vol = subset['last_sale_price'].sum()
    vol_share = (cluster_vol / total_realized_volume) * 100 if total_realized_volume > 0 else 0
    venduti = subset[subset['last_sale_price'] > 0]
    avg_price = venduti['last_sale_price'].mean() if not venduti.empty else 0
    median_price = venduti['last_sale_price'].median() if not venduti.empty else 0
    max_price = subset['last_sale_price'].max()

    #linguistic metrics
    avg_len = subset['length'].mean()
    crypto_perc = (subset['has_crypto'].mean() * 100) if 'has_crypto' in subset.columns else 0
    top_ngrams = get_top_ngrams(subset['Asset'], n=4, top_k=10)

    #top10 whales
    top_expensive = subset_sorted.head(10)['Asset'].tolist()
    #10 random examples
    random_samples = subset.sample(min(10, size))['Asset'].tolist()

    #report for each cluster
    print(f"Cluster {cid} | Size: {size} | 💎 Market Share: {vol_share:.1f}%")
    print(f"Average length: {avg_len:.1f} | Crypto presence: {crypto_perc:.1f}% | Average paid price: {avg_price:,.0f} TON")
    print(f"Top 4-grams: {top_ngrams}")
    print(f"Top 10 whales: {top_expensive}")
    print(f"10 random samples: {random_samples}")
    print(f"Max price paid: {max_price:,.0f} TON | Median: {median_price:,.0f} TON")
    print(" ")
    print(" ")

In [ ]:
#mapping (leaving here the original mapping, but has to be personalized)
short_taxonomy = {
    4: "5L Phonetic", 0: "4/5L Semantic", 2: "Tech Finance",
    8: "VIP & Gambling", 18: "Numerical Names", 16: "Media & Streaming",
    7: "Crypto & Memes", 19: "Crypto Utility", 11: "Asian Business 8L",
    12: "Brands & Foundations", 3: "Public Sector", 6: "Brands & Entertainment",
    13: "Money & Lifestyle", 17: "Repetition Patterns", 5: "Finance & Betting",
    15: "Big Tech", 1: "Chinese Mix", 9: "Generic Services",
    10: "Sports & Generic", 14: "Generic Phonetic"
}

df['cluster_label'] = df['linguistic_cluster'].map(short_taxonomy)

#top clusters by volume
summary = df.groupby('cluster_label')['last_sale_price'].sum().sort_values(ascending=False)
print(summary.head(10))
print("")

#check for most common linguistic cluster in the top 15 social communities
top_communities = df['social_community'].value_counts().head(15).index.tolist()
for comm_id in top_communities:
    subset = df[df['social_community'] == comm_id]

    if 'linguistic_cluster' in subset.columns and not subset['linguistic_cluster'].isna().all():
        top_cluster = subset['linguistic_cluster'].value_counts().idxmax()
        top_share = subset['linguistic_cluster'].value_counts(normalize=True).max() * 100
    else:
        top_cluster, top_share = "N/A", 0

    nome_comm = comm_names.get(comm_id, f"Community {comm_id}") if 'comm_names' in globals() else f"Community {comm_id}"
    print(f"{nome_comm} (Most frequent cluster in community {comm_id}): {top_cluster} ({top_share:.1f}%)")

# Mappings

In [ ]:
#distributional mapping

top_communities = df['social_community'].value_counts().head(15).index
df_plot = df[df['social_community'].isin(top_communities)]

heatmap_data = pd.crosstab(
    df_plot['social_community'],
    df_plot['cluster_label'],
    normalize='index'
) * 100

#order by cluster popularity
column_order = heatmap_data.mean().sort_values(ascending=False).index
heatmap_data = heatmap_data[column_order]

annot_labels = heatmap_data.applymap(lambda x: f"{x:.0f}%")

plt.figure(figsize=(22, 12))
sns.heatmap(
    heatmap_data,
    annot=annot_labels,
    fmt='',
    cmap='YlGnBu',
    linewidths=.8,
    linecolor='white',
    cbar=False,
    annot_kws={"size": 22}
)

plt.ylabel('')
plt.xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=24)
plt.yticks(rotation=45, fontsize=22)

plt.tight_layout()
plt.show()

In [ ]:
#average last sale price mapping

if 'cluster_label' not in df.columns:
    df['cluster_label'] = df['linguistic_cluster'].map(short_taxonomy)

top_communities = df['social_community'].value_counts().head(15).index
df_plot = df[df['social_community'].isin(top_communities)].copy()

#fill last_sale_price nulls with 0, so that they are included in the mean
df_plot['last_sale_price'] = df_plot['last_sale_price'].fillna(0)

avg_price_matrix = df_plot.pivot_table(
    index='social_community',
    columns='cluster_label',
    values='last_sale_price',
    aggfunc='mean'
).fillna(0)

#sort by descending value
col_order = avg_price_matrix.mean().sort_values(ascending=False).index
avg_price_matrix = avg_price_matrix[col_order]

def format_price(x):
    if x >= 1000:
        return f"{x/1000:.1f}k"
    elif x > 0:
        return f"{x:.0f}"
    else:
        return "0"

annot_labels = avg_price_matrix.applymap(format_price)
vmax_val = np.percentile(avg_price_matrix.values, 95)

plt.figure(figsize=(22, 12))
sns.heatmap(
    avg_price_matrix,
    annot=annot_labels,
    fmt='',
    cmap='YlOrRd',
    linewidths=.8,
    linecolor='white',
    vmin=0,
    vmax=vmax_val,
    cbar=False,
    annot_kws={"size": 22}
)

plt.ylabel('')
plt.xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=24)
plt.yticks(rotation=45, fontsize=22)

plt.tight_layout()
plt.show()

In [ ]:
#zero cost transfers mapping

top_communities = df['social_community'].value_counts().head(15).index
df_plot = df[df['social_community'].isin(top_communities)].copy()

#calculate event transfers
def count_transfers_only(history):
    transfers = 0
    if isinstance(history, list):
        for event in history:
            if isinstance(event, dict):
                price = event.get('Price')
                try:
                    if price is None or float(price) == 0:
                        transfers += 1
                except:
                    transfers += 1
    return transfers

df['event_transfers'] = df['parsed_history'].apply(count_transfers_only)

free_events_matrix = df_plot.pivot_table(
    index='social_community',
    columns='cluster_label',
    values='event_transfers',
    aggfunc='sum'
).fillna(0)

#order by zero cost volume
col_order = free_events_matrix.sum().sort_values(ascending=False).index
free_events_matrix = free_events_matrix[col_order]

plt.figure(figsize=(22, 12))
sns.heatmap(
    free_events_matrix,
    annot=True,
    fmt='.0f',
    cmap='Greens',
    linewidths=.8,
    linecolor='white',
    cbar=False,
    annot_kws={"size": 22}
)

plt.ylabel('')
plt.xlabel('')
plt.xticks(rotation=45, ha='right', fontsize=24)
plt.yticks(rotation=45, fontsize=22)

plt.tight_layout()
plt.show()

In [ ]:
from datetime import datetime

#average wallet holding time mapping

def parse_history(x):
    if pd.isna(x):
        return []
    try:
        return ast.literal_eval(x)
    except:
        return []

df['parsed_history'] = df['Ownership history'].apply(parse_history)

#compute average holding time
def compute_avg_wallet_holding_fixed(history):
    if not history:
        return 0

    dates = []
    for event in history:
        try:
            dates.append(pd.to_datetime(event['Date of Sale'], errors='coerce'))
        except:
            continue

    dates = sorted([d for d in dates if pd.notna(d)])

    if not dates:
        return 0

    #in case of only one sale, holding up to today
    if len(dates) == 1:
        delta = (pd.Timestamp.today() - dates[0]).days
        return max(delta, 0)

    #in case of more than one sale, mean between dates
    holding_periods = [(dates[i+1] - dates[i]).days for i in range(len(dates)-1) if (dates[i+1] - dates[i]).days >= 0]

    if not holding_periods:
        return 0

    return np.mean(holding_periods)

df['avg_wallet_holding_days'] = df['parsed_history'].apply(compute_avg_wallet_holding_fixed)

if 'cluster_label' not in df.columns:
    df['cluster_label'] = df['linguistic_cluster'].map(short_taxonomy)

top_communities = df['social_community'].value_counts().head(15).index
df_plot = df[df['social_community'].isin(top_communities)].copy()

holding_matrix = df_plot.pivot_table(
    index='social_community',
    columns='cluster_label',
    values='avg_wallet_holding_days',
    aggfunc='mean'
).fillna(0)

#order by global holding time
col_order = holding_matrix.mean().sort_values(ascending=False).index
holding_matrix = holding_matrix[col_order]

annot_labels = holding_matrix.applymap(lambda x: f"{x:.0f}")
vmax_val = np.percentile(holding_matrix.values, 95)

plt.figure(figsize=(22, 12))

sns.heatmap(
    holding_matrix,
    annot=annot_labels,
    fmt='',
    cmap='viridis',
    linewidths=.8,
    linecolor='white',
    vmin=0,
    vmax=vmax_val,
    cbar=False,
    annot_kws={"size": 22}
)

plt.ylabel('')
plt.xlabel('')

plt.xticks(rotation=45, ha='right', fontsize=24)
plt.yticks(rotation=45, fontsize=22)

plt.tight_layout()
plt.show()

# Bonus: linguistic clusters dendrogram

In [ ]:
#linguistic clusters dendrogram

plt.figure(figsize=(16, 10))

dendrogram(
    Z,
    labels=labels,
    orientation='top',
    leaf_rotation=90,
    leaf_font_size=24,
    color_threshold=None,
    above_threshold_color='black',
    distance_sort='descending',
    show_leaf_counts=True
)

plt.yticks(rotation=45, fontsize=22)
plt.tight_layout()
plt.show()